# Attention language model

Start from the same Tiny Shakespeare token stream as the bigram model, then inspect token embeddings before adding attention.

In [1]:
import random
import sys
from pathlib import Path

import torch
from torch import nn

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.dataset import get_batch, load_tiny_shakespeare_tokens, split_token_stream

## Prepare token streams and batches

In [2]:
tokens, vocab, merges = load_tiny_shakespeare_tokens(repo_root / "data")
train_tokens, validation_tokens = split_token_stream(tokens)

vocab_size = len(vocab)
block_size = 8
batch_size = 32
n_embd = 32
head_size = 16

random.seed(42)
x_batch, y_batch = get_batch(
    "train", train_tokens, validation_tokens, block_size, batch_size
)
x_batch = torch.tensor(x_batch, dtype=torch.long)
y_batch = torch.tensor(y_batch, dtype=torch.long)

## Model scaffold

In [3]:
class AttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        token_embeddings = self.token_embedding_table(idx)
        return self.lm_head(token_embeddings)


model = AttentionLanguageModel(vocab_size, n_embd)
token_embeddings = model.token_embedding_table(x_batch)
print(f"Token embedding shape (B, T, C): {tuple(token_embeddings.shape)}")

Token embedding shape (B, T, C): (32, 8, 32)


## TODO — first attention mechanics

1. Add positional embeddings so each token position has its own learned representation.
2. Implement one causal self-attention head over the `(B, T, C)` token embeddings.

Stop here before adding either piece.

In [4]:
# Positional embeddings
positional_embeddings = nn.Parameter(torch.zeros(1, block_size, n_embd))
# Add positional embeddings to token embeddings
token_embeddings += positional_embeddings
print(f"Token embedding shape after adding positional embeddings (B, T, C): {tuple(token_embeddings.shape)}")

Token embedding shape after adding positional embeddings (B, T, C): (32, 8, 32)


In [ ]:
# Causal self-attention
import math
import torch
from torch import nn

key = nn.Linear(n_embd, head_size, bias=False)
query = nn.Linear(n_embd, head_size, bias=False)
value = nn.Linear(n_embd, head_size, bias=False)

# x       [B, T, C]
# q,k,v   [B, T, H]
# scores  [B, T, T]
# weights [B, T, T]
# out     [B, T, H]
# MATCH THE SHAPES OF THE LINEAR LAYERS!! 
def causal_self_attention(x, mask=None):
    k = key(x)
    q = query(x)
    v = value(x)

    scores = q @ k.transpose(-1, -2) / math.sqrt(head_size)
    scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    attention_out = weights @ v
    return attention_out

x = torch.randn(1, 8, n_embd)
mask = torch.ones(1, 8, 8)
causal_self_attention(x, mask)

tensor([[[-0.1866, -0.0619,  0.2078,  0.3094, -0.1876,  0.2538,  0.1175,
           0.3606,  0.0683, -0.1984, -0.2348, -0.0614, -0.2062,  0.3063,
           0.0547,  0.1815],
         [-0.1992, -0.2019,  0.1511,  0.3085, -0.1240,  0.2607,  0.1597,
           0.4419,  0.0388, -0.2382, -0.3130, -0.2494, -0.1584,  0.3324,
          -0.0031,  0.1204],
         [-0.0231,  0.0122,  0.2737,  0.0134, -0.0454,  0.2391,  0.0675,
           0.3586, -0.1654, -0.3173, -0.2770, -0.2036, -0.3613,  0.3071,
           0.1202,  0.1122],
         [-0.2011, -0.1407,  0.1503,  0.3337, -0.1846,  0.2096,  0.0477,
           0.3965,  0.0707, -0.2098, -0.2326, -0.1037, -0.1495,  0.3116,
           0.0398,  0.1777],
         [-0.1354, -0.0628,  0.3001,  0.0631, -0.0753,  0.2431,  0.1663,
           0.3941, -0.1493, -0.2301, -0.1759, -0.2317, -0.1795,  0.2959,
           0.1080,  0.0518],
         [-0.1687,  0.0836,  0.4159, -0.0140, -0.0782,  0.3493,  0.3330,
           0.3333, -0.1575, -0.0914, -0.0978, -0.124